# Step 1 Inventory: SMH
Profiles every file at the site (columns, types, rows, ID and date formats) and records what later steps must be told: candidate keys, files that look like copies of one another, free-text columns, and years where a file's rows collapse. Nothing is changed. Outputs go to `derived/` and are counts, shapes and column names only.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.home() / "Documents/DS217/users/jormond/open_picu/code"))
import inventory as I

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 200)

In [ ]:
site = "SMH"                      # "SMH" | "GOSH"
demo = "ai_picu_demographics.csv" # GOSH: "PICANet_ALL_patients_SR_011024_projectID_eprweights.csv"
exclude = None                    # GOSH: r"lab_components"

base = Path.home() / "Documents/DS217/users/jormond"
open_picu_dir = base / "open_picu"
derived_dir = open_picu_dir / "derived"
reports_dir = derived_dir / "reports"
site_dir = base / site

# GOSH only: I.list_files(site_dir) first, then add recursive=True, n_jobs=3
inv = I.inventory_site(site_dir, exclude=exclude)
I.save_inventory(inv, derived_dir / "inventory" / f"{site.lower()}_inventory.pkl")
inv["summary"]

## 1. Structure
Files with identical columns are grouped (one file per variable is common at GOSH). Candidate keys are every ID-like column; "unique per row" tells you which could identify a row. ID overlap is the share of each file's IDs that appear in the base file.

In [ ]:
I.schema_groups(inv)

In [ ]:
I.id_overlap(inv, base=demo)

## 2. Dates and eras
Format, range and resolution of every date column, then rows per calendar year by each file's main time column. `find_gaps` lists years below half of the file's usual level. These are leads to explain (system changeover, outage, pandemic), not errors. `rows_by_year` rereads one column per file, so allow a few minutes at GOSH.

In [ ]:
inv["dates"]

In [ ]:
by_year = I.rows_by_year(inv)
by_year

In [ ]:
I.find_gaps(by_year)

## 3. Privacy flags and look-alike files
Columns flagged as free text or direct identifiers are carried forward: no later step may release a column with one of these names without a decision. Look-alike files are candidates to settle at step 2.

In [ ]:
I.identifier_audit(inv)

In [ ]:
I.lookalike_files(inv)

## 4. Decisions and notes
The only place judgements are typed. `rejected` lists files that must not be used and why. `notes` are short statements, with no values or IDs, that later steps should know.

In [ ]:
# RBH
# rejected = {"ai_picu_piccanet_old.csv": "old PICANet file; not used (manifest rule 2)"}
# notes = ["studyID = prefix (NIC/NND) : PATIENT_RK : SAS-second admission time; 5 stays appear under both prefixes",
#          "NND rows admitted before 2010 mostly have no vitals (7,917 empty studyIDs)",
#          "demographics holds 14,797 studyIDs, 1,139 admissions have more than one studyID"]

# SMH
rejected = {}
notes = ["encounterId is the stay key; UniquePtID is not equal between demographics and piccanet, so do not link on it",
         "large exact-duplicate row counts in vitals, vent, airway, io (see summary)"]

# GOSH
# rejected = {"Archived/picanet_file_with_mapping.csv": "unreliable discharge status; not used",
#             "PICANet_All_patients_SR_011024_projectID.csv": "identical to the eprweights file; eprweights used"}
# notes = ["links by project_id; admission_identifier on about 56% of HR rows only",
#          "Epic era from 2019-04-20: see GOSH_EPIC_ERA.md"]

In [ ]:
paths = I.finalise(inv, site, reports_dir, base=demo, by_year=by_year,
                   rejected=rejected, excluded=exclude, notes=notes)
paths

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
sys.path.append(str(Path.home() / "Documents/DS217/users/jormond/open_picu"))
import pandas as pd
import code.inventory as I
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 200)

In [ ]:
base = Path.home() / "Documents/DS217/users/jormond"
open_picu_dir = base / "open_picu"
smh_dir = base / "SMH"

smh = I.inventory_site(smh_dir)   # add max_rows=2_000_000 for a first pass
smh["summary"]

In [ ]:
smh["dates"]

In [ ]:
I.identifier_audit(smh)

In [ ]:
u = pd.read_csv(smh_dir/"ai_picu_infusion.csv", usecols=["utcChartTime"], low_memory=False)["utcChartTime"]
print(I.shapes_of(u).value_counts().head(8))

In [ ]:
I.write_report(smh, "SMH", open_picu_dir/"SMH_inventory.md")

In [ ]:
for fn, prof in smh["profiles"].items():
    print(fn)
    display(prof)

In [ ]:
display(I.shared_columns(smh))
display(I.id_overlap(smh))

In [ ]:
I.write_report(smh, "SMH", open_picu_dir/"SMH_inventory.md")

In [ ]:
d = pd.read_csv(smh_dir/"ai_picu_demographics.csv")
d1 = d.drop_duplicates()
print(len(d), len(d1), "unique encounters:", d1["encounterId"].nunique(), "patients:", d1["UniquePtID"].nunique())
dup = d1[d1["encounterId"].duplicated(keep=False)]
print(dup["encounterId"].nunique(), "encounterIds with differing rows; columns that differ:")
print(dup.groupby("encounterId").nunique().max())
print("encounters per patient:\n", d1.groupby("UniquePtID")["encounterId"].nunique().value_counts().head(8))

In [ ]:
pn = pd.read_csv(smh_dir/"ai_picu_piccanet.csv")
print("PICANet rows:", len(pn), "| unique encounterId:", pn["encounterId"].nunique(),
      "| in demographics:", round(pn["encounterId"].isin(d1["encounterId"]).mean(), 3),
      "| PIM3 missing:", int(pn["PIM3"].isna().sum()))
m = pn.merge(d1[["encounterId", "UniquePtID"]].drop_duplicates(), on="encounterId", suffixes=("", "_dem"))
print("UniquePtID equal in both files:", round((m["UniquePtID"] == m["UniquePtID_dem"]).mean(), 3))

d1 = d1.assign(year=I.parse_dates(d1["inTime"])[0].dt.year,
               picanet=d1["encounterId"].isin(pn["encounterId"]))
print(pd.crosstab(d1["year"], d1["picanet"]))

In [ ]:
for fn in ["ai_picu_vitals.csv", "ai_picu_vent.csv", "ai_picu_airway.csv"]:
    x = pd.read_csv(smh_dir/fn, usecols=["interventionId", "terseForm"])
    print(fn)
    print(x.groupby("interventionId")["terseForm"].agg(
        n="size", top=lambda s: s.value_counts().head(5).to_dict()).to_string())

In [ ]:
v = pd.read_csv(smh_dir/"ai_picu_vitals.csv", usecols=["encounterId", "utcChartTime"])
first = pd.to_datetime(v["utcChartTime"]).groupby(v["encounterId"]).min()
t_in = pd.Series(I.parse_dates(d1["inTime"])[0].values, index=d1["encounterId"].values)
t_in = t_in[~t_in.index.duplicated()]
dh = (first - t_in.reindex(first.index)).dt.total_seconds() / 3600
print(dh.describe())
print(dh.round().value_counts().head(8))

In [ ]:
g = d1.groupby("encounterId")
print("encounters with >1 inTime:", int((g["inTime"].nunique() > 1).sum()),
      "| with >1 Weight:", int((g["Weight"].nunique() > 1).sum()))
d1 = d1.assign(t=I.parse_dates(d1["inTime"])[0], w=pd.to_numeric(d1["Weight"], errors="coerce"))
print("non-numeric Weight entries:", int(d1["Weight"].notna().sum() - d1["w"].notna().sum()))
gap = d1.groupby("encounterId")["t"].agg(lambda s: (s.max() - s.min()).total_seconds() / 3600)
print("inTime spread within an encounter (h), where > 0:\n", gap[gap > 0].describe())
ratio = d1.groupby("encounterId")["w"].agg(lambda s: s.max() / s.min())
print("max/min weight within an encounter, where > 1:\n", ratio[ratio > 1].describe())

In [ ]:
print(d1.groupby(["year", "picanet"])["encounterId"].nunique().unstack(fill_value=0))
print("unique encounters:", d1["encounterId"].nunique())

In [ ]:
mo = t_in.reindex(dh.index).dt.month
print("May-Sep:", dh[mo.between(5, 9)].round().value_counts().head(4).to_dict())
print("Nov-Feb:", dh[(mo >= 11) | (mo <= 2)].round().value_counts().head(4).to_dict())

In [ ]:
pw = pn.drop_duplicates("encounterId").set_index("encounterId")["Weight"].rename("pn_weight")
dw = d1.groupby("encounterId")["w"].agg(["min", "median", "max"])
c = dw.join(pw, how="inner").dropna(subset=["pn_weight"])
print(len(c), "encounters with both")
for col in ["min", "median", "max"]:
    ok = ((c[col] - c["pn_weight"]).abs() / c["pn_weight"] < 0.05).mean()
    print(col, "within 5% of PICANet weight:", round(ok, 3))